In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../dataset/ac97_top/features/net_features_place.csv', sep=',')

In [3]:
df_y = pd.read_csv('../dataset/ac97_top/route_label/net_labels.csv', sep=',')

In [4]:
df_all = df.merge(df_y, on='net_name', how='inner')

In [5]:
#df_all['label']
print("features nets:", df['net_name'].nunique())
print("label nets:", df_y['net_name'].nunique())
print("merged nets:", df_all['net_name'].nunique())

features nets: 7318
label nets: 8351
merged nets: 7315


In [6]:
df_all['label'].value_counts()

label
0    6742
1     573
Name: count, dtype: int64

In [7]:
df_all

,net_name,status,bbox_area,bbox_aspect,bbox_cx,bbox_cy,bbox_dx,bbox_dy,bbox_perimeter,bbox_xmax,...,sink_quadrant_q1,sink_quadrant_q2,sink_quadrant_q3,sink_quadrant_q4,sink_x_mean,sink_x_std,sink_y_mean,sink_y_std,term_count,label
0,_0000_,OK,148.730526,1.294952,53.6510,474.5515,13.878,10.717,49.190,60.590,...,1,0,0,0,58.420000,NaN,480.080000,NaN,2,0
1,_0001_,OK,4.991328,0.771226,48.0310,652.7180,1.962,2.544,9.012,49.012,...,1,0,0,0,49.220000,NaN,654.160000,NaN,2,0
2,_0002_,OK,4.368874,0.675580,64.1310,538.6415,1.718,2.543,8.522,64.990,...,0,0,0,1,67.160000,NaN,537.200000,NaN,2,0
3,_0003_,OK,38.687968,0.214690,64.5910,620.0780,2.882,13.424,32.612,66.032,...,1,0,0,0,65.550000,NaN,626.960000,NaN,2,0
4,_0004_,OK,480.895074,0.291411,53.0910,693.6815,11.838,40.623,104.922,59.010,...,0,0,0,1,61.180000,NaN,673.200000,NaN,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7310,net5239,OK,4182.392680,0.277048,29.8770,437.9195,34.040,122.867,313.814,46.897,...,4,3,3,1,29.523636,10.760851,450.407273,34.453531,12,0
7311,net5241,OK,3427.561200,0.302501,26.6570,250.1890,32.200,106.446,277.292,42.757,...,3,2,3,3,29.565455,11.580994,246.654545,28.452486,12,0
7312,net5240,OK,5591.167638,0.204076,32.0465,364.5140,33.779,165.522,398.602,48.936,...,3,3,2,4,33.695000,10.137627,362.213333,52.028765,13,0
7313,net5244,OK,2852.736000,0.240993,28.2670,251.3660,26.220,108.800,270.040,41.377,...,4,0,4,3,32.450909,8.021246,241.956364,31.451302,12,0


In [8]:
# какие колонки точно не фичи
non_feature_cols = [
    'net_name',
    'net_dbid',
    'odb_net',
    'status'
]

x = df_all.drop(columns=[c for c in non_feature_cols if c in df_all.columns] + ['label'])
y = df_all['label']

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_hpwl = df_all[['is_bus_bit']]
y = df_all['label']

Xtr, Xte, ytr, yte = train_test_split(
    X_hpwl, y, test_size=0.3, random_state=42, stratify=y
)

clf = LogisticRegression(class_weight='balanced')
clf.fit(Xtr, ytr)

proba = clf.predict_proba(Xte)[:,1]
print("ROC-AUC (is_bus_bit):", roc_auc_score(yte, proba))

ROC-AUC (is_bus_bit): 0.6666431962661945


In [10]:
df_all.columns

Index(['net_name', 'status', 'bbox_area', 'bbox_aspect', 'bbox_cx', 'bbox_cy',
       'bbox_dx', 'bbox_dy', 'bbox_perimeter', 'bbox_xmax', 'bbox_xmin',
       'bbox_ymax', 'bbox_ymin', 'bterm_count', 'dist_drv_sink_max',
       'dist_drv_sink_mean', 'dist_drv_sink_sum', 'driver_inst_area',
       'driver_is_buf', 'driver_is_clkbuf', 'driver_is_ff_q', 'driver_is_inv',
       'fanout', 'has_bterm', 'has_multiple_drivers', 'hpwl', 'is_bus_bit',
       'is_clock_like', 'is_enable_like', 'is_reg2reg_candidate',
       'is_reset_like', 'iterm_count', 'net_dbid', 'num_buf_drivers',
       'num_buf_sinks', 'num_clkbuf_drivers', 'num_drivers', 'num_ff_sinks_d',
       'num_inout', 'num_inv_drivers', 'num_inv_sinks', 'num_port_drivers',
       'num_port_sinks', 'num_sinks', 'odb_net', 'sink_area_buf_sum',
       'sink_area_ff_sum', 'sink_area_inv_sum', 'sink_area_max',
       'sink_area_mean', 'sink_area_std', 'sink_area_sum',
       'sink_centroid_dist_to_driver', 'sink_cnt', 'sink_dist_p50',
 

In [24]:
#df_new = df_all.drop(columns=['net_name','status'])
#df_new.corr()['label']

In [11]:
non_feature_cols = [
    'net_name',
    'net_dbid',
    'odb_net',
    'status'
]

df_feat = df_all.drop(
    columns=[c for c in non_feature_cols if c in df_all.columns],
    errors='ignore'
)

In [12]:
for c in df_feat.columns:
    df_feat[c] = pd.to_numeric(df_feat[c], errors='coerce')

In [13]:
df_feat = df_feat.dropna(axis=1, how='all')

In [14]:
nunique = df_feat.nunique()
const_cols = nunique[nunique <= 1].index.tolist()

print(f"Drop constant columns ({len(const_cols)}):")
print(const_cols)

df_feat = df_feat.drop(columns=const_cols)

Drop constant columns (5):
['driver_is_clkbuf', 'has_multiple_drivers', 'num_clkbuf_drivers', 'num_drivers', 'num_inout']


In [15]:
label = df_feat['label']
X = df_feat.drop(columns=['label'])

In [16]:
corr_spearman = X.corrwith(label, method='spearman')
corr_spearman = corr_spearman.sort_values(key=lambda x: x.abs(), ascending=False)

corr_spearman.head(20)

is_bus_bit         -0.190965
sink_min_x          0.180557
bbox_xmin           0.179954
bbox_cx             0.178593
sink_x_mean         0.177595
bbox_xmax           0.175446
sink_max_x          0.173593
num_ff_sinks_d      0.114197
sink_area_ff_sum    0.112391
num_buf_drivers    -0.076373
driver_is_buf      -0.076373
driver_is_ff_q     -0.070298
sink_area_max      -0.066243
sink_y_std          0.056955
sink_area_mean     -0.052027
sink_area_sum      -0.047363
num_port_sinks      0.046602
sink_x_std          0.046311
sink_dist_p50       0.041743
sink_quadrant_q3   -0.040880
dtype: float64

In [39]:
df_feat.corr()['label']

bbox_area     -0.018829
bbox_aspect    0.000421
bbox_cx        0.176204
bbox_cy        0.019319
bbox_dx       -0.018302
                 ...   
sink_x_std    -0.028762
sink_y_mean    0.017532
sink_y_std     0.028975
term_count    -0.007085
label          1.000000
Name: label, Length: 62, dtype: float64

In [41]:
corr_spearman = X.corrwith(label, method='spearman')
corr_spearman = corr_spearman.sort_values(key=lambda x: x.abs(), ascending=False)

corr_spearman.head(20)

corr_pearson = X.corrwith(label, method='pearson')
corr_pearson = corr_pearson.sort_values(key=lambda x: x.abs(), ascending=False)

corr_df = pd.DataFrame({
    'spearman': corr_spearman,
    'pearson': corr_pearson
})

corr_df['abs_spearman'] = corr_df['spearman'].abs()
corr_df = corr_df.sort_values('abs_spearman', ascending=False)

corr_df.head(30)

,spearman,pearson,abs_spearman
is_bus_bit,-0.190965,-0.190965,0.190965
sink_min_x,0.180557,0.178053,0.180557
bbox_xmin,0.179954,0.177175,0.179954
bbox_cx,0.178593,0.176204,0.178593
sink_x_mean,0.177595,0.175005,0.177595
bbox_xmax,0.175446,0.172558,0.175446
sink_max_x,0.173593,0.170841,0.173593
num_ff_sinks_d,0.114197,0.114197,0.114197
sink_area_ff_sum,0.112391,0.112665,0.112391
num_buf_drivers,-0.076373,-0.076373,0.076373


In [64]:
X = df_feat.drop(columns='label')
Y = label
X

,bbox_area,bbox_aspect,bbox_cx,bbox_cy,bbox_dx,bbox_dy,bbox_perimeter,bbox_xmax,bbox_xmin,bbox_ymax,...,sink_min_y,sink_quadrant_q1,sink_quadrant_q2,sink_quadrant_q3,sink_quadrant_q4,sink_x_mean,sink_x_std,sink_y_mean,sink_y_std,term_count
0,148.730526,1.294952,53.6510,474.5515,13.878,10.717,49.190,60.590,46.712,479.910,...,480.08,1,0,0,0,58.420000,NaN,480.080000,NaN,2
1,4.991328,0.771226,48.0310,652.7180,1.962,2.544,9.012,49.012,47.050,653.990,...,654.16,1,0,0,0,49.220000,NaN,654.160000,NaN,2
2,4.368874,0.675580,64.1310,538.6415,1.718,2.543,8.522,64.990,63.272,539.913,...,537.20,0,0,0,1,67.160000,NaN,537.200000,NaN,2
3,38.687968,0.214690,64.5910,620.0780,2.882,13.424,32.612,66.032,63.150,626.790,...,626.96,1,0,0,0,65.550000,NaN,626.960000,NaN,2
4,480.895074,0.291411,53.0910,693.6815,11.838,40.623,104.922,59.010,47.172,713.993,...,673.20,0,0,0,1,61.180000,NaN,673.200000,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7310,4182.392680,0.277048,29.8770,437.9195,34.040,122.867,313.814,46.897,12.857,499.353,...,376.72,4,3,3,1,29.523636,10.760851,450.407273,34.453531,12
7311,3427.561200,0.302501,26.6570,250.1890,32.200,106.446,277.292,42.757,10.557,303.412,...,197.20,3,2,3,3,29.565455,11.580994,246.654545,28.452486,12
7312,5591.167638,0.204076,32.0465,364.5140,33.779,165.522,398.602,48.936,15.157,447.275,...,281.52,3,3,2,4,33.695000,10.137627,362.213333,52.028765,13
7313,2852.736000,0.240993,28.2670,251.3660,26.220,108.800,270.040,41.377,15.157,305.766,...,197.20,4,0,4,3,32.450909,8.021246,241.956364,31.451302,12


In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [66]:
pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos

scale_pos_weight

np.float64(11.768079800498754)

In [67]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

In [68]:
xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [69]:
from sklearn.metrics import roc_auc_score

proba_test = xgb.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, proba_test)

roc_auc

0.9534107760751359

In [70]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, proba_test)
pr_auc

0.6915470293761786

In [71]:
def recall_at_k(y_true, y_score, k):
    idx = np.argsort(y_score)[::-1][:k]
    return y_true.iloc[idx].sum() / y_true.sum()

k = y_test.sum()
recall_k = recall_at_k(y_test, proba_test, int(k))

recall_k

np.float64(0.6627906976744186)

In [72]:
import pandas as pd

imp = pd.Series(
    xgb.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

imp.head(20)

is_bus_bit           0.236336
num_ff_sinks_d       0.046901
has_bterm            0.039507
driver_inst_area     0.035187
driver_is_ff_q       0.031429
sink_cnt             0.027994
num_port_sinks       0.027310
driver_is_buf        0.026081
iterm_count          0.025817
sink_area_max        0.024023
bbox_xmin            0.022916
sink_area_ff_sum     0.022539
sink_area_sum        0.020508
bterm_count          0.020095
fanout               0.018901
sink_min_x           0.018240
sink_x_std           0.018137
sink_x_mean          0.017610
bbox_xmax            0.015726
dist_drv_sink_sum    0.015343
dtype: float32